In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from sklearn.metrics import accuracy_score

print('####################################################################')
print('                 classifying online toxic comments                  ')
print('####################################################################')
print('Choose one of the following options to continue: ')
print('1- Classify dataset of comments using LSTM model')
print('2- Split data and find the accuracy')
print('3- classify  dataset of comments  using keras ')
print('4- load a pretrain keras model ')
print('5- Classify a single comment using LSTM model')
print('--------------------------------------------------------------------')

# Helper function for loading and processing data
def load_data(train_path, test_path=None):
    train_data = pd.read_csv(train_path, encoding='utf-8', on_bad_lines='skip').fillna(' ')
    X_train = train_data["comment_text"]
    Y_train = train_data[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
    if test_path:
        test_data = pd.read_csv(test_path, encoding='utf-8', on_bad_lines='skip').fillna(' ')
        X_test = test_data["comment_text"]
        return X_train, Y_train, X_test
    return X_train, Y_train, None

# Tokenization and Padding
def preprocess_text(X_train, X_test=None, max_features=20000, max_len=200):
    tokenizer = Tokenizer(num_words=max_features)

    # Fit the tokenizer on the training text
    tokenizer.fit_on_texts(X_train)

    # Tokenize and pad the training text
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_train_padded = pad_sequences(X_train_seq, maxlen=max_len)

    # Check for any zero-length sequences (comments that became empty)
    valid_indexes_train = [i for i, x in enumerate(X_train_seq) if len(x) > 0]

    # Filter out empty sequences from both X_train_padded and Y_train
    X_train_padded = X_train_padded[valid_indexes_train]

    if X_test is not None:
        # Tokenize and pad the test text
        X_test_seq = tokenizer.texts_to_sequences(X_test)
        X_test_padded = pad_sequences(X_test_seq, maxlen=max_len)

        # Check for any zero-length sequences in X_test
        valid_indexes_test = [i for i, x in enumerate(X_test_seq) if len(x) > 0]
        X_test_padded = X_test_padded[valid_indexes_test]

        return X_train_padded, X_test_padded, tokenizer, valid_indexes_train, valid_indexes_test

    return X_train_padded, tokenizer, valid_indexes_train

# Build LSTM Model
def build_lstm_model(input_length, num_classes):
    model = Sequential([
        Embedding(input_dim=20000, output_dim=128),
        LSTM(60, return_sequences=True),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(50, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='sigmoid')  # Use 'sigmoid' for multi-label classification
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Training and Evaluation
def train_and_evaluate_model(X_train, Y_train, X_test, model):
    model.fit(X_train, Y_train, epochs=10, batch_size=256, validation_split=0.1)
    y_pred = model.predict(X_test)
    return y_pred

while True:
    option = int(input("What do you want to do (choose 1, 2, 3, 4, 5 or 0 to exit): "))

    if option == 1:
        X_train, Y_train, X_test = load_data(input("Train data path: "), input("Test data path: "))
        X_train_padded, X_test_padded, tokenizer, valid_indexes_train, valid_indexes_test = preprocess_text(X_train, X_test)
        Y_train_filtered = Y_train.iloc[valid_indexes_train].reset_index(drop=True)
        num_classes = Y_train_filtered.shape[1]

        # Build and train the model
        model = build_lstm_model(input_length=X_train_padded.shape[1], num_classes=num_classes)
        y_pred = train_and_evaluate_model(X_train_padded, Y_train_filtered, X_test_padded, model)
        result_final = pd.concat([X_test.reset_index(drop=True), pd.DataFrame(y_pred, columns=Y_train.columns)], axis=1)
        result_final.to_csv(input("Save result as CSV file (name): ") + '.csv', index=False)

        # Convert predictions to binary (0 or 1) based on a threshold of 0.5
        y_pred_binary = (y_pred > 0.5).astype(int)

        # Count the occurrences of each category
        category_counts = y_pred_binary.sum(axis=0)

        # Print the summary
        print("\nClassification Summary:")
        for category, count in zip(Y_train.columns, category_counts):
            print(f"{category}: {count} comments")

    elif option == 2:
        # Load and Predict with the Keras Model
        X_train, Y_train, _ = load_data(input("Train data path: "))
        comment = input("Write comment to classify: ")
        X_train_padded, tokenizer, valid_indexes_train = preprocess_text(X_train)
        Y_train_filtered = Y_train.iloc[valid_indexes_train].reset_index(drop=True)
        X_test_padded, _, _ = preprocess_text(pd.Series([comment]), max_features=20000, max_len=200)

        # Build and train the model
        model = build_lstm_model(input_length=200, num_classes=Y_train_filtered.shape[1])
        model.fit(X_train_padded, Y_train_filtered, epochs=10, batch_size=8192, validation_split=0.1)

        y_prediction = model.predict(X_test_padded)
        print(f'The result for the comment "{comment}":')
        print(y_prediction)

    elif option == 3:
        X_train, Y_train, _ = load_data(input("Train data path: "))
        X_train_padded, tokenizer, valid_indexes_train = preprocess_text(X_train)
        Y_train_filtered = Y_train.iloc[valid_indexes_train].reset_index(drop=True)
        num_classes = Y_train_filtered.shape[1]

        # Split the data
        X_train_split, X_test_split, Y_train_split, Y_test_split = train_test_split(X_train_padded, Y_train_filtered, test_size=0.2, random_state=42)

        # Build and train the model
        model = build_lstm_model(input_length=X_train_padded.shape[1], num_classes=num_classes)
        y_pred = train_and_evaluate_model(X_train_split, Y_train_split, X_test_split, model)

        # Calculate accuracy
        accuracy = accuracy_score(Y_test_split, (y_pred > 0.5).astype(int))
        print("The accuracy score =", accuracy)

    elif option == 4:
        # Build and Save the Model
        num_classes = 6  # Replace with the actual number of classes
        #The input_length is set to 200 which is the max_len used in preprocess_text function.
        model = build_lstm_model(input_length = 200, num_classes=num_classes)
        # Build the model by calling it with dummy data
        model.build((None, 200))  # Adjust the input shape as needed
        model.save(input("Save model as: ") + '.keras')  # Save the model in .keras format
    elif option == 5:
        # Load and predict with the Keras model
        model_path = input("Load Keras model (file path): ")
        model = tf.keras.models.load_model(model_path)  # No need to add .keras here
        comment = input("Input comment to predict: ")
        X_test_padded, tokenizer, _ = preprocess_text(pd.Series([comment]), max_features=20000, max_len=200) # Modified line to unpack three values
        y_prediction = model.predict(X_test_padded)
        print("Prediction result:", y_prediction)

    elif option == 0:
        print("Exiting the program.")
        break

    else:
        print("Invalid option. Please choose from the list or type 0 to exit.")

####################################################################
                 classifying online toxic comments                  
####################################################################
Choose one of the following options to continue: 
1- Classify dataset of comments using LSTM model
2- Split data and find the accuracy
3- classify  dataset of comments  using keras 
4- load a pretrain keras model 
5- Classify a single comment using LSTM model
--------------------------------------------------------------------
What do you want to do (choose 1, 2, 3, 4, 5 or 0 to exit): 1
Train data path: /content/train.csv
Test data path: /content/test.csv
Epoch 1/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.4615 - loss: 0.2234 - val_accuracy: 0.9940 - val_loss: 0.0582
Epoch 2/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.9391 - loss: 0.0597 - val_accuracy: 0.9940 - val_loss: 0.0512
Epoch 3/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9739

/usr/local/lib/python3.10/dist-packages/keras/src/saving/saving_lib.py:576: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Input comment to predict: you are very shamefull person
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Prediction result: [[0.50692403 0.5029439  0.49591392 0.5018947  0.49656737 0.49875772]]
What do you want to do (choose 1, 2, 3, 4, 5 or 0 to exit): 0
Exiting the program.
